In [1]:
import ast
import os
from Bio import SeqIO
import pandas as pd
from tqdm import tqdm
import multiprocessing
import warnings
import time
import random

warnings.filterwarnings('ignore')

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
finder_dir = '/home/zhongshitong/Chromid_finder'
os.system(f'python {finder_dir}/Chromid-finder_run.py -h')

usage: Chromid-finder_run.py [-h] -i INPUT -n CPU -o OUTPUT -d DT

Run Chromid-finder pipeline

optional arguments:
  -h, --help            show this help message and exit
  -i INPUT, --input INPUT
                        Input fasta file
  -n CPU, --cpu CPU     Number of CPUs
  -o OUTPUT, --output OUTPUT
                        Output file
  -d DT, --dt DT        Parameter for dt


0

In [3]:
def chromid_finding(acc_n, temp_folder_queue, que):
    result_dir = f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/chromid_results'
    if os.path.exists(f'{result_dir}/{acc_n}.txt'):
        que.put(1)
        return
    os.makedirs(result_dir, exist_ok=True)
    temp_folder = temp_folder_queue.get()
    
    os.chdir(temp_folder)
    
    gbff_path = f'/data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff'
    temp_fasta = f'temp_nucleotide_{acc_n}.fasta'
    with open(gbff_path, 'r') as handle, open(temp_fasta, 'w') as temp_file:
        seq_records = SeqIO.parse(handle, 'genbank')
        SeqIO.write(seq_records, temp_file, "fasta")
    
    os.system(f'python Chromid-finder_run.py -i {temp_fasta} -o {result_dir}/{acc_n}.txt -n 2 -d 1.6 >/dev/null 2>&1')
    
    if os.path.exists(temp_fasta):
        os.remove(temp_fasta)
        
    temp_folder_queue.put(temp_folder)
    que.put(1)

In [4]:
par = 32
temp_folder_queue = multiprocessing.Manager().Queue()
temp_folders = []
for i in range(par):
    op_path = f'/active-data/temp/Chromid_finder_{i}'
    temp_folders.append(op_path)
    os.system(f'cp -r {finder_dir} {op_path}')
    temp_folder_queue.put(op_path)

for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    acc_list = org_data_n['accession'].tolist()

    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    tot = len(acc_list)
    pool = multiprocessing.Pool(par)
    
    for acc_n in acc_list:
        pool.apply_async(
            chromid_finding,
            args=(acc_n, temp_folder_queue, que)
        )
    
    pool.close()
    
    with tqdm(total = len(acc_list), desc=f'{genus_name}({len(acc_list)})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        count = 0
        while True:
            time.sleep(0.01)
            if not que.empty():
                temp_data = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
    
    pool.join()

for i in range(par):
    op_path = f'/active-data/temp/Chromid_finder_{i}'
    os.system(f'rm -r {op_path}')

Escherichia(4204): 100%|██████████████████████████████████████| 4.20k/4.20k [2:24:59<00:00, 2.07s/B]
Klebsiella(3554): 100%|███████████████████████████████████████| 3.55k/3.55k [2:32:23<00:00, 2.57s/B]
Staphylococcus(2423): 100%|█████████████████████████████████████| 2.42k/2.42k [24:47<00:00, 1.63B/s]
Pseudomonas(2343): 100%|██████████████████████████████████████| 2.34k/2.34k [2:09:09<00:00, 3.31s/B]
Bacillus(1976): 100%|███████████████████████████████████████████| 1.98k/1.98k [45:31<00:00, 1.38s/B]
Salmonella(1853): 100%|███████████████████████████████████████| 1.85k/1.85k [1:06:55<00:00, 2.17s/B]
Streptococcus(1599): 100%|██████████████████████████████████████| 1.60k/1.60k [15:53<00:00, 1.68B/s]
Streptomyces(1359): 100%|███████████████████████████████████████| 1.36k/1.36k [40:29<00:00, 1.79s/B]
Acinetobacter(1234): 100%|██████████████████████████████████████| 1.23k/1.23k [19:29<00:00, 1.05B/s]
Helicobacter(416): 100%|████████████████████████████████████████████| 416/416 [03:34<00:00,